## Technology Parameters Dataset

Create `technologies_2030.csv` combining data from multiple sources:

**Data Sources:**
- **Capacity data**: `pemmdb_capacities_2030_grouped.csv` (69 unique `index_carrier` technologies)
- **Cost parameters**: `costs_2030_processed.csv` (efficiency, VOM, FOM from PyPSA technology-data)
- **Fuel prices**: TYNDP 2024 Study values (€/GJ → €/MWh via ×3.6)
- **CO2 emissions**: TYNDP 2024 Study values (kgCO2/GJ → tCO2/MWh via ×3.6/1000)

**Dataset Structure:**
- `index_carrier`: Specific technology from PEMMDB (69 types)
- `carrier`: Generic technology group (types: chp, gas, coal, onwind, etc.)
- `pypsa_carrier`: Network carrier for PyPSA (types: separates renewables, groups fossils by emission profile)
- `fuel_type`: Base fuel being burned (for fuel prices: coal, gas, lignite, oil-*, h2, nuclear, renewable)
- `emission_type`: Emission profile (for CO2: separates CCS variants like coal-ccs, gas-ccs)
- `costs_technology`: Technology name in costs CSV for parameter lookup

**Key TYNDP 2024 Study Values:**
- **Fuel prices (€/GJ)**: Nuclear 1.7, Coal 1.8, Gas 6.3, Oil-heavy 9.6, Oil-light 11.7, H2 17.6
- **CO2 emissions (kgCO2/GJ)**: 
  - Gas CCS: **5.70** (90% capture)
  - Gas (all non-CCS): **51.57**
  - Coal: 94.0, Coal-CCS: 9.4
  - Lignite: 101.0, Lignite-CCS: 10.1
- **CO2 price**: 113.4 €/tCO2

**Marginal Cost Calculation:**
```
MC = VOM + (fuel_price / efficiency) + (CO2_emissions / efficiency) × CO2_price
```

**CCS Technology Handling:**
- `fuel_type` = base fuel (e.g., 'coal' for coal-ccs) → used for fuel prices
- `emission_type` = technology variant (e.g., 'coal-ccs') → used for CO2 emissions
- `pypsa_carrier` = emission_type (to distinguish CCS from non-CCS in network)

In [36]:
import pandas as pd
import numpy as np

capacities = pd.read_csv('../data/open-tyndp/pemmdb_capacities_2030_grouped.csv')
costs = pd.read_csv('../data/open-tyndp/costs_2030_processed.csv')

In [37]:
print(capacities["index_carrier"].unique())

['coal' 'coal-ccs' 'gas-ccgt' 'gas-ccgt-ccs' 'gas-conv' 'gas-ocgt'
 'lignite' 'lignite-ccs' 'nuclear' 'oil-heavy' 'oil-light' 'oil-shale'
 'electrolyser-onshore-grid' 'hydro-ror-turbine' 'hydro-pondage-reservoir'
 'hydro-pondage-turbine' 'hydro-reservoir-reservoir'
 'hydro-reservoir-turbine' 'hydro-phs-reservoir' 'hydro-phs-turbine'
 'hydro-phs-pump' 'hydro-phs-pure-reservoir' 'hydro-phs-pure-turbine'
 'hydro-phs-pure-pump' 'onwind' 'offwind' 'solar-thermal'
 'solar-pv-utility' 'solar-pv-rooftop' 'solar-thermal-w-storage'
 'solar-thermal-w-storage-storage' 'battery-charge' 'battery-discharge'
 'battery-store' 'h2-fuel-cell' 'h2-ccgt' 'other-res'
 'chp-gas-ccgt-old-1-other-70.65eur'
 'chp-gas-conventional-old-2-industrial-0eur'
 'chp-gas-conventional-old-1-industrial-0eur'
 'chp-gas-conventional-old-1-other-0eur' 'chp-lignite-old-1-other-0eur'
 'chp-gas-conventional-old-1-other-128.4eur'
 'chp-gas-ccgt-old-2-other-58.87500000000006eur'
 'chp-gas-ccgt-old-1-other-0eur' 'chp-gas-conventio

In [38]:
print(capacities["carrier"].unique())

['coal' 'gas' 'lignite' 'uranium' 'oil-heavy' 'oil-light' 'oil-shale'
 'electrolyser' 'hydro-ror' 'hydro-pondage' 'hydro-reservoir' 'hydro-phs'
 'hydro-phs-pure' 'onwind' 'offwind' 'solar-thermal' 'solar-pv-utility'
 'solar-pv-rooftop' 'solar-thermal-w-storage' 'battery' 'h2-fuel-cell'
 'h2-ccgt' 'other-res' 'chp']


In [39]:
# Get unique index_carriers with their carrier mapping
carrier_mapping = capacities[['index_carrier', 'carrier']].drop_duplicates().set_index('index_carrier')['carrier'].to_dict()
index_carriers = sorted(capacities['index_carrier'].unique())
print(f"Found {len(index_carriers)} unique index_carriers\n")

# TYNDP 2024 Study Data
study_fuel_prices = {'nuclear': 1.7, 'coal': 1.8, 'lignite': 1.8, 'gas': 6.3, 
                     'oil-heavy': 9.6, 'oil-light': 11.7, 'oil-shale': 1.9, 'h2': 17.6}

# CO2 emissions - includes both base fuels and CCS technologies
study_co2_emissions = {'nuclear': 0.0, 'coal': 94.0, 'coal-ccs': 9.4, 'lignite': 101.0, 
                       'lignite-ccs': 10.1, 'gas': 51.57, 'gas-ccs': 5.70,
                       'oil-light': 78.0, 'oil-heavy': 78.0, 'oil-shale': 100.0, 'h2': 0.0}

co2_price = 113.4  # €/tCO2

Found 69 unique index_carriers



In [40]:

def get_open_tyndp_type(idx_c, csv_value=None):
    """Return open_tyndp_type for a given index_carrier.
    
    Priority:
      1. Value already present in CSV (non-null) — trusted source for standard technologies
      2. Derived from index_carrier name — used for CHP/DSR where CSV has null
    
    Granularity hierarchy:
      carrier         (broadest)  →  gas
      open_tyndp_type             →  gas-ccgt         ← this
      index_carrier   (finest)    →  chp-gas-ccgt-old-1-other-0eur
    """
    if csv_value and not (isinstance(csv_value, float) and pd.isna(csv_value)):
        return csv_value  # Trust CSV value for standard technologies

    # Derive from name — handles CHP/DSR with price bands in the name
    if 'coal-ccs' in idx_c or 'hard-coal-ccs' in idx_c: return 'coal-ccs'
    if 'lignite-ccs' in idx_c: return 'lignite-ccs'
    if 'gas-ccgt-ccs' in idx_c: return 'gas-ccgt-ccs'
    if 'nuclear' in idx_c: return 'nuclear'
    if 'hard-coal' in idx_c or ('coal' in idx_c and 'lignite' not in idx_c): return 'coal'
    if 'lignite' in idx_c: return 'lignite'
    if 'gas-ccgt' in idx_c or 'gas-conv' in idx_c or 'gas-conventional' in idx_c: return 'gas-ccgt'
    if 'gas-ocgt' in idx_c: return 'gas-ocgt'
    if 'oil-light' in idx_c or 'light-oil' in idx_c: return 'oil-light'
    if 'oil-heavy' in idx_c or 'heavy-oil' in idx_c: return 'oil-heavy'
    if 'oil-shale' in idx_c: return 'oil-shale'
    if 'h2-ccgt' in idx_c: return 'h2-ccgt'
    if 'h2-fuel-cell' in idx_c: return 'h2-fuel-cell'
    if 'battery-charge' in idx_c: return 'battery-charge'
    if 'battery-discharge' in idx_c: return 'battery-discharge'
    if 'battery-store' in idx_c: return 'battery-store'
    if 'battery' in idx_c: return 'battery'
    if 'electrolyser' in idx_c: return 'electrolyser'
    if 'hydro-ror' in idx_c: return 'hydro-ror-turbine'
    if 'hydro-pondage' in idx_c and 'turbine' in idx_c: return 'hydro-pondage-turbine'
    if 'hydro-pondage' in idx_c: return 'hydro-pondage-reservoir'
    if 'hydro-reservoir' in idx_c and 'turbine' in idx_c: return 'hydro-reservoir-turbine'
    if 'hydro-reservoir' in idx_c: return 'hydro-reservoir-reservoir'
    if 'hydro-phs-pure' in idx_c and 'pump' in idx_c: return 'hydro-phs-pure-pump'
    if 'hydro-phs-pure' in idx_c and 'turbine' in idx_c: return 'hydro-phs-pure-turbine'
    if 'hydro-phs-pure' in idx_c: return 'hydro-phs-pure-reservoir'
    if 'hydro-phs' in idx_c and 'pump' in idx_c: return 'hydro-phs-pump'
    if 'hydro-phs' in idx_c and 'turbine' in idx_c: return 'hydro-phs-turbine'
    if 'hydro-phs' in idx_c: return 'hydro-phs-reservoir'
    if 'onwind' in idx_c: return 'onwind'
    if 'offwind' in idx_c: return 'offwind'
    if 'solar-pv-rooftop' in idx_c: return 'solar-pv-rooftop'
    if 'solar-pv-utility' in idx_c: return 'solar-pv-utility'
    if 'solar-thermal-w-storage-storage' in idx_c: return 'solar-thermal-w-storage-storage'
    if 'solar-thermal-w-storage' in idx_c: return 'solar-thermal-w-storage'
    if 'solar-thermal' in idx_c: return 'solar-thermal'
    if 'other-res' in idx_c: return 'other-res'
    if 'dsr' in idx_c: return 'dsr'
    if 'chp' in idx_c: return 'chp'
    return idx_c  # fallback: keep as-is


def get_pypsa_carrier(row):
    """Determine PyPSA carrier name for network definition"""
    # Special case: hydrogen technologies keep their specific carrier
    if row['index_carrier'] in ('h2-ccgt', 'h2-fuel-cell'):
        return row['index_carrier']
    if row['carrier'] == 'chp':
        return 'other-thermal'  # All CHP plants grouped as other-thermal
    elif row['fuel_type'] == 'renewable':
        return row['carrier']  # Use carrier: onwind, offwind, solar-pv-utility, battery, etc.
    else:
        return row['emission_type']  # Use emission_type: gas, coal, gas-ccs, coal-ccs, etc.


def get_params(idx_c):
    """Map index_carrier to (base_fuel_type, emission_type, costs_technology)"""
    # Fossil fuels with CCS
    if 'coal-ccs' in idx_c or 'hard-coal-ccs' in idx_c:
        return ('coal', 'coal-ccs', 'coal-ccs')
    if 'lignite-ccs' in idx_c:
        return ('lignite', 'lignite-ccs', 'lignite-ccs')
    if 'gas-ccgt-ccs' in idx_c:
        return ('gas', 'gas-ccs', 'gas-ccgt-ccs')

    # Fossil fuels without CCS
    if 'nuclear' in idx_c: return ('nuclear', 'nuclear', 'nuclear')
    if 'coal' in idx_c or 'hard-coal' in idx_c: return ('coal', 'coal', 'coal')
    if 'lignite' in idx_c: return ('lignite', 'lignite', 'lignite')
    if 'gas-ccgt' in idx_c or 'gas-conv' in idx_c or 'gas-conventional' in idx_c: return ('gas', 'gas', 'CCGT')
    if 'gas-ocgt' in idx_c: return ('gas', 'gas', 'OCGT')

    # Oil variants
    if 'oil-light' in idx_c or 'light-oil' in idx_c: return ('oil-light', 'oil-light', 'oil-light')
    if 'oil-heavy' in idx_c or 'heavy-oil' in idx_c: return ('oil-heavy', 'oil-heavy', 'oil-heavy')
    if 'oil-shale' in idx_c: return ('oil-shale', 'oil-shale', 'oil-shale')

    # Hydrogen
    if 'h2-ccgt' in idx_c: return ('h2', 'h2-ccgt', 'central hydrogen CHP')
    if 'h2-fuel-cell' in idx_c: return ('h2', 'h2-fuel-cell', 'fuel cell')

    # DSR — treat as load-side, no fuel
    if 'dsr' in idx_c: return (None, None, None)

    # Renewables
    if 'onwind' in idx_c: return (None, None, 'onwind')
    if 'offwind' in idx_c: return (None, None, 'offwind')
    if 'solar-pv-utility' in idx_c: return (None, None, 'solar')
    if 'solar-pv-rooftop' in idx_c: return (None, None, 'solar-rooftop')
    if 'solar-thermal' in idx_c: return (None, None, 'central solar thermal')
    if 'other-res' in idx_c: return (None, None, None)

    # Hydro
    if 'hydro' in idx_c: return (None, None, 'hydro')

    # Storage
    if 'battery-charge' in idx_c or 'battery-discharge' in idx_c:
        return (None, None, 'battery inverter')
    if 'battery-store' in idx_c or 'battery' in idx_c:
        return (None, None, 'battery storage')
    if 'electrolyser' in idx_c: return (None, None, 'electrolysis')

    return (None, None, None)


# Hardcoded marginal costs for specific carriers
HARDCODED_MC = {
    'other-res': 45.0,   # Biomass/waste opportunity cost
}

# Build carrier & open_tyndp_type mappings from granular CSV
# my_pemmdb_capacities_2030.csv has open_tyndp_type filled for standard techs;
# CHP/DSR rows have null — filled here via get_open_tyndp_type()
caps_unique = capacities[['index_carrier', 'carrier', 'open_tyndp_type']].drop_duplicates('index_carrier')
carrier_mapping      = caps_unique.set_index('index_carrier')['carrier'].to_dict()
open_tyndp_type_from_csv = caps_unique.set_index('index_carrier')['open_tyndp_type'].to_dict()

index_carriers = sorted(capacities['index_carrier'].dropna().unique())
print(f"Found {len(index_carriers)} unique index_carriers")

# Build dataset
data = []
missing = []

for idx_c in index_carriers:
    base_fuel, tech_type, cost_tech = get_params(idx_c)
    carrier = carrier_mapping.get(idx_c, 'unknown')
    open_tyndp_type = get_open_tyndp_type(idx_c, open_tyndp_type_from_csv.get(idx_c))

    is_storage = any(x in idx_c for x in ['reservoir', 'store'])
    is_pump = 'pump' in idx_c

    if cost_tech:
        costs_row = costs[costs['technology'] == cost_tech]
        if not costs_row.empty:
            eff = costs_row['efficiency'].iloc[0]
            vom = costs_row['VOM'].iloc[0]
            fom = costs_row['FOM'].iloc[0]
        else:
            if is_storage:
                eff, vom, fom = 1.0, 0.0, 0.0
            elif is_pump:
                eff, vom, fom = 0.9, 0.0, 1.0
            else:
                missing.append(idx_c)
                eff, vom, fom = 1.0, 0.0, 0.0
    else:
        eff, vom, fom = 1.0, 0.0, 0.0

    if base_fuel:
        fuel_price_gj = study_fuel_prices.get(base_fuel, 0)
        fuel_price_mwh = fuel_price_gj * 3.6
        co2_kg_gj = study_co2_emissions.get(tech_type, 0)
        co2_tco2_mwh = co2_kg_gj * 3.6 / 1000
        if eff > 0:
            mc = vom + (fuel_price_mwh / eff) + (co2_tco2_mwh / eff) * co2_price
        else:
            mc = vom
    else:
        fuel_price_gj, fuel_price_mwh = 0, 0
        co2_kg_gj, co2_tco2_mwh = 0, 0
        mc = vom

    if carrier in HARDCODED_MC:
        mc = HARDCODED_MC[carrier]

    data.append({
        'index_carrier': idx_c,
        'carrier': carrier,
        'open_tyndp_type': open_tyndp_type,
        'fuel_type': base_fuel if base_fuel else 'renewable',
        'emission_type': tech_type if tech_type else 'renewable',
        'costs_technology': cost_tech if cost_tech else '',
        'efficiency': eff if pd.notna(eff) else 1.0,
        'vom_eur_mwh': vom if pd.notna(vom) else 0.0,
        'fom_eur_mw_year': fom if pd.notna(fom) else 0.0,
        'fuel_price_eur_gj': fuel_price_gj,
        'fuel_price_eur_mwh': fuel_price_mwh,
        'co2_kg_gj': co2_kg_gj,
        'co2_tco2_mwh': co2_tco2_mwh,
        'marginal_cost_eur_mwh': mc if pd.notna(mc) else 0.0,
    })

df = pd.DataFrame(data)
df['pypsa_carrier'] = df.apply(get_pypsa_carrier, axis=1)

cols = ['index_carrier', 'carrier', 'open_tyndp_type', 'pypsa_carrier', 'fuel_type', 'emission_type',
        'costs_technology', 'efficiency', 'vom_eur_mwh', 'fom_eur_mw_year',
        'fuel_price_eur_gj', 'fuel_price_eur_mwh', 'co2_kg_gj', 'co2_tco2_mwh',
        'marginal_cost_eur_mwh']
df = df[cols]

df.to_csv('../data/open-tyndp/technologies_2030.csv', index=False)

print(f"Created dataset with {len(df)} technologies")
print(f"{len(missing)} technologies not found in costs CSV")
if missing:
    print(f"  Missing: {missing}\n")

# Show open_tyndp_type coverage
null_ott = df['open_tyndp_type'].isna().sum()
print(f"\nopen_tyndp_type null: {null_ott} / {len(df)}")

print("\nUnique PyPSA carriers:")
unique_carriers = sorted(df['pypsa_carrier'].unique())
print(f"Total: {len(unique_carriers)}")
print(unique_carriers)

print("\nHardcoded MC verification:")
for carrier_name, expected_mc in HARDCODED_MC.items():
    rows = df[df['carrier'] == carrier_name]
    if not rows.empty:
        actual = rows['marginal_cost_eur_mwh'].iloc[0]
        status = "✓" if actual == expected_mc else "✗"
        print(f"  {status} {carrier_name}: {actual} €/MWh (expected {expected_mc})")


Found 69 unique index_carriers
Created dataset with 69 technologies
0 technologies not found in costs CSV

open_tyndp_type null: 0 / 69

Unique PyPSA carriers:
Total: 27
['battery', 'coal', 'coal-ccs', 'electrolyser', 'gas', 'gas-ccs', 'h2-ccgt', 'h2-fuel-cell', 'hydro-phs', 'hydro-phs-pure', 'hydro-pondage', 'hydro-reservoir', 'hydro-ror', 'lignite', 'lignite-ccs', 'nuclear', 'offwind', 'oil-heavy', 'oil-light', 'oil-shale', 'onwind', 'other-res', 'other-thermal', 'solar-pv-rooftop', 'solar-pv-utility', 'solar-thermal', 'solar-thermal-w-storage']

Hardcoded MC verification:
  ✓ other-res: 45.0 €/MWh (expected 45.0)


In [41]:
import sys
sys.path.insert(0, '.')
from helpers import CARRIERS

unique_set   = set(unique_carriers)
helpers_set  = set(CARRIERS)

only_in_unique  = sorted(unique_set  - helpers_set)
only_in_helpers = sorted(helpers_set - unique_set)
in_both         = sorted(unique_set  & helpers_set)

print(f"unique_carriers (pypsa_carrier from CSV) : {len(unique_set)}")
print(f"CARRIERS in helpers.py                   : {len(helpers_set)}")
print()
print(f"✓ In BOTH ({len(in_both)}):")
for c in in_both:
    print(f"   {c}")

print(f"\n⚠  Only in unique_carriers — NOT in CARRIERS ({len(only_in_unique)}):")
for c in only_in_unique:
    print(f"   {c}")

print(f"\n⚠  Only in CARRIERS — NOT in unique_carriers ({len(only_in_helpers)}):")
for c in only_in_helpers:
    print(f"   {c}")


unique_carriers (pypsa_carrier from CSV) : 27
CARRIERS in helpers.py                   : 28

✓ In BOTH (26):
   battery
   coal
   coal-ccs
   gas
   gas-ccs
   h2-ccgt
   h2-fuel-cell
   hydro-phs
   hydro-phs-pure
   hydro-pondage
   hydro-reservoir
   hydro-ror
   lignite
   lignite-ccs
   nuclear
   offwind
   oil-heavy
   oil-light
   oil-shale
   onwind
   other-res
   other-thermal
   solar-pv-rooftop
   solar-pv-utility
   solar-thermal
   solar-thermal-w-storage

⚠  Only in unique_carriers — NOT in CARRIERS (1):
   electrolyser

⚠  Only in CARRIERS — NOT in unique_carriers (2):
   chp
   uranium


In [42]:
import sys
sys.path.insert(0, '.')
from helpers import CARRIERS, CARRIERS_TO_SIMPLE

carriers_set = set(CARRIERS)
simple_keys  = set(CARRIERS_TO_SIMPLE.keys())

missing_in_simple = sorted(carriers_set - simple_keys)

print(f"Keys in CARRIERS but missing from CARRIERS_TO_SIMPLE ({len(missing_in_simple)}):")
for c in missing_in_simple:
    print(f"  {repr(c)}")


Keys in CARRIERS but missing from CARRIERS_TO_SIMPLE (1):
  'uranium'
